# Rossmann Store Sales — Data Cleaning

## Objective

Clean and standardize the raw Rossmann datasets based on the data quality issues identified during the data understanding stage.

## Cleaning Principles

- Keep the raw datasets unchanged.
- Apply only evidence-based cleaning decisions.
- Validate every transformation after applying it.
- Treat structural missingness differently from unexpected missingness.
- Avoid unnecessary transformations before modeling requirements are known.

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = RAW_DATA_DIR / "train.csv"
STORE_PATH = RAW_DATA_DIR / "store.csv"

In [ ]:
Path.cwd()

In [ ]:
Path.cwd().parent

In [ ]:
TRAIN_PATH.exists(), STORE_PATH.exists()

## 2. Load Raw Data

Load the original datasets without modifying the raw files. Separate working copies will be created for cleaning.

In [ ]:
train = pd.read_csv(TRAIN_PATH, low_memory=False)
store = pd.read_csv(STORE_PATH)

In [ ]:
train.shape, store.shape

In [ ]:
train_clean = train.copy()
store_clean = store.copy()

## 3. Standardize `StateHoliday`

`StateHoliday` is a categorical variable. During the initial data-understanding stage, Pandas' default CSV type inference produced mixed Python types for this column.

When loading the dataset with `low_memory=False`, the values are consistently interpreted as strings. The column is explicitly converted to Pandas' `string` dtype to enforce a clear and consistent schema.

In [ ]:
train_clean["StateHoliday"].value_counts(dropna=False)

In [ ]:
train_clean["StateHoliday"].map(type).value_counts()

In [ ]:
train_clean["StateHoliday"] = train_clean["StateHoliday"].astype("string")

In [ ]:
train_clean["StateHoliday"].dtype

In [ ]:
train_clean["StateHoliday"].unique()

In [ ]:
train_clean["StateHoliday"].value_counts(dropna=False)

In [ ]:
train_clean["StateHoliday"].nunique(dropna=False)

## 4. Convert `Date` to Datetime

The `Date` column represents temporal information but is loaded from the CSV as a string.

It will be converted to Pandas' datetime dtype so that dates can be correctly sorted, compared, validated, and used in later time-based analysis.

In [ ]:
train_clean["Date"].dtype

In [ ]:
train_clean["Date"] = pd.to_datetime(
    train_clean["Date"], format="%Y-%m-%d", errors="raise"
)

In [ ]:
train_clean["Date"].dtype

In [ ]:
train_clean["Date"].isna().sum()

In [ ]:
train_clean["Date"].min(), train_clean["Date"].max()

In [ ]:
train_clean["Date"].head()

### Date Cleaning Validation

- The `Date` column was successfully converted to datetime.
- No missing dates were introduced during conversion.
- The original date range was preserved: 2013-01-01 to 2015-07-31.
- Multiple rows share the same date because each observation represents a specific store on a specific day.
- The `00:00:00` time component is expected because the original data contains dates without time information.

## 5. Standardize Store Metadata Data Types

Several store metadata columns are stored as `float64` because they contain missing values, even though their values represent discrete quantities such as months, years, and weeks.

Before converting them to nullable integer types, verify that all observed values are integer-like.

In [ ]:
store_clean.dtypes

In [ ]:
integer_like_columns = [
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2SinceWeek",
    "Promo2SinceYear",
]

for column in integer_like_columns:
    non_missing = store_clean[column].dropna()
    non_integer_count = ((non_missing % 1) != 0).sum()

    print(column, non_integer_count)

In [ ]:
for column in integer_like_columns:
    store_clean[column] = store_clean[column].astype("Int64")

In [ ]:
store_clean[integer_like_columns].dtypes

In [ ]:
store_clean[integer_like_columns].isna().sum()

In [ ]:
store_clean[
    [
        "CompetitionOpenSinceMonth",
        "CompetitionOpenSinceYear",
        "Promo2SinceWeek",
        "Promo2SinceYear",
    ]
].head(10)

### Store Metadata Type Cleaning

The competition month/year and Promo2 week/year columns contained only integer-like observed values but were stored as `float64` because of missing values.

These columns were converted to Pandas' nullable `Int64` dtype so that their discrete meaning is preserved while retaining missing values.

### Integer Type Conversion Validation

- All four discrete metadata columns were successfully converted to nullable `Int64`.
- Missing-value counts remained unchanged after conversion.
- Integer-like values are now represented without unnecessary decimal points.
- Missing values are preserved as nullable integer values rather than being imputed.

## 6. Validate Missing-Value Semantics

In [ ]:
promo2_metadata_columns = [
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval",
]

In [ ]:
promo2_inactive = store_clean["Promo2"].eq(0)

store_clean.loc[
    promo2_inactive,
    promo2_metadata_columns,
].isna().all()

In [ ]:
promo2_active = store_clean["Promo2"].eq(1)

store_clean.loc[
    promo2_active,
    promo2_metadata_columns,
].isna().sum()

### Promo2 Missingness Decision

Promo2-related missing values are structural rather than unexpected missing data.

- All stores with `Promo2 = 0` have missing values in `Promo2SinceWeek`, `Promo2SinceYear`, and `PromoInterval`.
- All stores with `Promo2 = 1` have complete Promo2 metadata.
- These missing values represent "not applicable" rather than unknown information.
- No imputation is applied to Promo2 metadata at this stage.

### Competition Missingness

In [ ]:
competition_columns = [
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
]

In [ ]:
store_clean[competition_columns].isna().value_counts()

### Competition Missingness Decision

Competition-related missing values are not imputed during data cleaning.

For 351 stores, `CompetitionDistance` is available while the competition opening month and year are unknown. Filling these fields with median values would create artificial historical information and could distort future duration-based features.

For 3 additional stores, all competition-related information is missing.

Therefore, competition missingness is preserved at this stage and will be handled explicitly during feature engineering or modeling if necessary.

- 761 stores have complete competition information.
- 351 stores have a known `CompetitionDistance` but unknown competition opening month and year.
- 3 stores are missing all competition-related metadata.
- These missing values are preserved because imputing opening dates would create artificial historical information.

## 7. Standardize Categorical Metadata Types

In [ ]:
categorical_store_columns = [
    "StoreType",
    "Assortment",
    "PromoInterval",
]

In [ ]:
store_clean[categorical_store_columns].dtypes

In [ ]:
for column in categorical_store_columns:
    store_clean[column] = store_clean[column].astype("string")

In [ ]:
store_clean[categorical_store_columns].dtypes

In [ ]:
store_clean[categorical_store_columns].isna().sum()

### Categorical Type Conversion Validation

- `StoreType`, `Assortment`, and `PromoInterval` were successfully converted to Pandas' `string` dtype.
- Missing-value counts remained unchanged after conversion.
- `StoreType` and `Assortment` contain no missing values.
- `PromoInterval` retains 544 structural missing values associated with stores where `Promo2 = 0`.

In [ ]:
store_clean.loc[
    store_clean["CompetitionDistance"].isna(),
    [
        "Store",
        "CompetitionDistance",
        "CompetitionOpenSinceMonth",
        "CompetitionOpenSinceYear",
        "StoreType",
        "Assortment",
        "Promo2",
    ],
]

### Competition Distance Missingness Decision

Three stores have missing `CompetitionDistance` together with missing competition opening month and year.

No direct evidence is available to recover their true competition distance. Replacing these values with the median would only provide a plausible central value, not the actual historical information.

Therefore, `CompetitionDistance` missing values are preserved during data cleaning and will be handled later during model preprocessing if required.

## 8. Final Cleaning Validation

In [ ]:
train.shape, train_clean.shape

In [ ]:
store.shape, store_clean.shape

In [ ]:
train_clean.duplicated().sum()

In [ ]:
store_clean.duplicated().sum()

In [ ]:
train_clean.dtypes

In [ ]:
store_clean.dtypes

In [ ]:
train_clean.isna().sum()

In [ ]:
store_clean.isna().sum()

In [ ]:
set(train_clean["Store"]) == set(store_clean["Store"])

In [ ]:
train_clean.duplicated(subset=["Store", "Date"]).sum()

### Final Cleaning Validation

- Dataset shapes were preserved during cleaning.
- No duplicate rows were introduced.
- Each `Store`-`Date` combination remains unique.
- `train.csv` and `store.csv` still contain the same set of 1,115 stores.
- `Date` was converted to a datetime dtype.
- Text-based categorical columns were standardized to Pandas' `string` dtype.
- Discrete month, year, and week metadata were converted to nullable `Int64`.
- Structural Promo2 missing values were intentionally preserved.
- Competition-related missing values were intentionally preserved rather than imputed.
- No rows were removed during cleaning.

## 9. Save Cleaned Data

The cleaned datasets are saved in Parquet format to preserve column types and provide efficient storage for downstream analysis.

In [ ]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CLEAN_PATH = PROCESSED_DATA_DIR / "train_clean.parquet"
STORE_CLEAN_PATH = PROCESSED_DATA_DIR / "store_clean.parquet"

In [ ]:
train_clean.to_parquet(
    TRAIN_CLEAN_PATH,
    index=False,
)

store_clean.to_parquet(
    STORE_CLEAN_PATH,
    index=False,
)

In [ ]:
train_processed = pd.read_parquet(TRAIN_CLEAN_PATH)
store_processed = pd.read_parquet(STORE_CLEAN_PATH)

In [ ]:
train_processed.shape, store_processed.shape

In [ ]:
train_processed.dtypes

In [ ]:
store_processed.dtypes

In [ ]:
train_processed.equals(train_clean)

In [ ]:
store_processed.equals(store_clean)

### Processed Data Validation

- Both cleaned datasets were successfully written to Parquet and reloaded.
- Dataset shapes were preserved after the write-read round trip.
- Column data types were preserved.
- Reloaded datasets are exactly equal to their in-memory cleaned versions.